## 准备数据

In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, optimizers, datasets

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'  # or any {'0', '1', '2'}

def mnist_dataset():
    (x, y), (x_test, y_test) = datasets.mnist.load_data()
    #normalize
    x = x/255.0
    x_test = x_test/255.0
    
    return (x, y), (x_test, y_test)

In [2]:
print(list(zip([1, 2, 3, 4], ['a', 'b', 'c', 'd'])))

[(1, 'a'), (2, 'b'), (3, 'c'), (4, 'd')]


## 建立模型

In [16]:
class myModel:
    def __init__(self):
        ####################
        '''声明模型对应的参数'''
        self.W1 = tf.Variable(tf.random.normal([28*28, 100]))
        self.b1 = tf.Variable(tf.zeros([100]))
        self.W2 = tf.Variable(tf.random.normal([100, 10]))
        self.b2 = tf.Variable(tf.zeros([10]))
        
        ####################
    def __call__(self, x):
        ####################
        '''实现模型函数体，返回未归一化的logits'''
        ####################
        
        x = tf.reshape(x, [-1, 28*28])
        h = tf.nn.relu(tf.matmul(x, self.W1) + self.b1)
        logits = tf.matmul(h, self.W2) + self.b2
        return logits
        
model = myModel()

optimizer = optimizers.Adam()

## 计算 loss

In [17]:
@tf.function
def compute_loss(logits, labels):
    return tf.reduce_mean(
        tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels))

@tf.function
def compute_accuracy(logits, labels):
    predictions = tf.argmax(logits, axis=1)
    return tf.reduce_mean(tf.cast(tf.equal(predictions, labels), tf.float32))

@tf.function
def train_one_step(model, optimizer, x, y):
    with tf.GradientTape() as tape:
        logits = model(x)
        loss = compute_loss(logits, y)

    # compute gradient
    trainable_vars = [model.W1, model.W2, model.b1, model.b2]
    grads = tape.gradient(loss, trainable_vars)
    optimizer.apply_gradients(zip(grads, trainable_vars))

    accuracy = compute_accuracy(logits, y)

    # loss and accuracy is scalar tensor
    return loss, accuracy

@tf.function
def test(model, x, y):
    logits = model(x)
    loss = compute_loss(logits, y)
    accuracy = compute_accuracy(logits, y)
    return loss, accuracy

## 实际训练

In [18]:
train_data, test_data = mnist_dataset()
for epoch in range(50):
    loss, accuracy = train_one_step(model, optimizer, 
                                    tf.constant(train_data[0], dtype=tf.float32), 
                                    tf.constant(train_data[1], dtype=tf.int64))
    print('epoch', epoch, ': loss', loss.numpy(), '; accuracy', accuracy.numpy())
loss, accuracy = test(model, 
                      tf.constant(test_data[0], dtype=tf.float32), 
                      tf.constant(test_data[1], dtype=tf.int64))

print('test loss', loss.numpy(), '; accuracy', accuracy.numpy())

epoch 0 : loss 114.27555 ; accuracy 0.08771667
epoch 1 : loss 111.416466 ; accuracy 0.0892
epoch 2 : loss 108.660614 ; accuracy 0.0904
epoch 3 : loss 106.006516 ; accuracy 0.09245
epoch 4 : loss 103.446365 ; accuracy 0.0945
epoch 5 : loss 100.97234 ; accuracy 0.09728333
epoch 6 : loss 98.578735 ; accuracy 0.09975
epoch 7 : loss 96.259834 ; accuracy 0.10168333
epoch 8 : loss 94.00768 ; accuracy 0.10355
epoch 9 : loss 91.81612 ; accuracy 0.10555
epoch 10 : loss 89.67983 ; accuracy 0.10745
epoch 11 : loss 87.5932 ; accuracy 0.1092
epoch 12 : loss 85.55271 ; accuracy 0.11125
epoch 13 : loss 83.5544 ; accuracy 0.113233335
epoch 14 : loss 81.59638 ; accuracy 0.11505
epoch 15 : loss 79.67722 ; accuracy 0.11716667
epoch 16 : loss 77.79398 ; accuracy 0.118933335
epoch 17 : loss 75.94587 ; accuracy 0.12111667
epoch 18 : loss 74.13336 ; accuracy 0.1228
epoch 19 : loss 72.356766 ; accuracy 0.1249
epoch 20 : loss 70.61591 ; accuracy 0.12708333
epoch 21 : loss 68.91113 ; accuracy 0.12933333
epoch 22